In [74]:
import numpy as np
import pandas as pd
import os, re, csv
from dotenv import load_dotenv

import warnings
warnings.filterwarnings('ignore')

load_dotenv()

True

In [99]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import word_tokenize, sent_tokenize
import string
from string import punctuation

In [25]:
def cleanData(text, stemming = False, lemmatize=False):    
    text = text.lower().split()
    text = " ".join(text)
    text = re.sub(r"[^A-Za-z0-9^,!.\/'+\-=]", " ", text)
    text = re.sub(r"what's", "what is ", text)
    text = re.sub(r"\'s", " ", text)
    text = re.sub(r"\'ve", " have ", text)
    text = re.sub(r"can't", "cannot ", text)
    text = re.sub(r"n't", " not ", text)
    text = re.sub(r"i'm", "i am ", text)
    text = re.sub(r"\'re", " are ", text)
    text = re.sub(r"\'d", " would ", text)
    text = re.sub(r"\'ll", " will ", text)
    text = re.sub(r",", " ", text)
    text = re.sub(r"\.", " ", text)
    text = re.sub(r"!", " ! ", text)
    text = re.sub(r"\/", " ", text)
    text = re.sub(r"\^", " ^ ", text)
    text = re.sub(r"\+", " + ", text)
    text = re.sub(r"\-", " - ", text)
    text = re.sub(r"\=", " = ", text)
    text = re.sub(r"'", " ", text)
    text = re.sub(r"(\d+)(k)", r"\g<1>000", text)
    text = re.sub(r":", " : ", text)
    text = re.sub(r" e g ", " eg ", text)
    text = re.sub(r" b g ", " bg ", text)
    text = re.sub(r" u s ", " american ", text)
    text = re.sub(r"\0s", "0", text)
    text = re.sub(r" 9 11 ", "911", text)
    text = re.sub(r"e - mail", "email", text)
    text = re.sub(r"j k", "jk", text)
    text = re.sub(r"\s{2,}", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    if stemming:
        st = PorterStemmer()
        txt = " ".join([st.stem(w) for w in text.split()])
    if lemmatize:
        wordnet_lemmatizer = WordNetLemmatizer()
        txt = " ".join([wordnet_lemmatizer.lemmatize(w) for w in text.split()])
    return text

In [26]:
special_character_removal=re.compile(r'[^a-z\d ]',re.IGNORECASE)
replace_numbers=re.compile(r'\d+',re.IGNORECASE)

def text_to_wordlist(text, remove_stopwords=False, stem_words=False):
    text = text.lower().split()
    
    text = " ".join(text)
    
    text=special_character_removal.sub('',text)
    text=replace_numbers.sub('',text)
    
    return(text)

In [27]:
train_df = pd.read_csv(os.getenv('train_csv'))
test_df = pd.read_csv(os.getenv('test_csv'))

train_df['comment_text'] = train_df['comment_text'].map(lambda x: cleanData(x))
test_df['comment_text'] = test_df['comment_text'].map(lambda x: cleanData(x))

In [34]:
classes = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

comments_train = train_df['comment_text'].values
comments_test = test_df['comment_text'].values

In [35]:
sentences_train=[]
for text in comments_train:
    sentences_train.append(text_to_wordlist(text))

sentences_test=[]
for text in comments_test:
    sentences_test.append(text_to_wordlist(text))

In [36]:
sentences_train[1]

'd aww  he matches this background colour i am seemingly stuck with thanks talk   january   utc'

In [33]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.layers import *
from keras.models import Model
from tensorflow.keras.layers import BatchNormalization

In [65]:
MAX_SEQUENCE_LENGTH = 150
MAX_NB_WORDS = 100000
EMBEDDING_DIM = 128
VALIDATION_SPLIT = 0.1

In [41]:
tokenizer =  Tokenizer(num_words=MAX_NB_WORDS)
tokenizer.fit_on_texts(sentences_train + sentences_test)

In [59]:
seq_train = tokenizer.texts_to_sequences(sentences_train)
seq_test = tokenizer.texts_to_sequences(sentences_test)

X = pad_sequences(
    seq_train,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    seq_test,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding='post',
    truncating='post'
)

y = train_df[classes]

In [60]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VALIDATION_SPLIT, random_state=68
)

In [56]:
import pickle

with open('tokenizer_r.pkl','wb') as f:
    pickle.dump(tokenizer,f)

In [61]:
print("Training Shape :", X_train.shape)
print("Validation Shape :", X_val.shape)
print("Test Shape :", X_test.shape)

print("Training Labels :", y_train.shape)
print("Validation Labels :", y_val.shape)

Training Shape : (143613, 150)
Validation Shape : (15958, 150)
Test Shape : (153164, 150)
Training Labels : (143613, 6)
Validation Labels : (15958, 6)


In [63]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout
from tensorflow.keras.layers import SimpleRNN, LSTM, Bidirectional

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [64]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'working_model.keras',
    monitor='val_loss',
    save_best_only=True
)

In [73]:
# SIMPLE RNN
model_rnn = Sequential()
model_rnn.add(
    Embedding(
        input_dim=MAX_NB_WORDS,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH
    )
)
model_rnn.add(SimpleRNN(128))
model_rnn.add(Dropout(0.3))
model_rnn.add(Dense(64, activation='relu'))
model_rnn.add(Dense(6, activation='sigmoid'))

model_rnn.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy'])

In [67]:
history_rnn = model_rnn.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/10
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 175s 154ms/step - accuracy: 0.9113 - loss: 0.1456 - val_accuracy: 0.9937 - val_loss: 0.1318
Epoch 2/10
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 179s 159ms/step - accuracy: 0.9815 - loss: 0.1322 - val_accuracy: 0.9842 - val_loss: 0.1403
Epoch 3/10
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 177s 158ms/step - accuracy: 0.9793 - loss: 0.1401 - val_accuracy: 0.9937 - val_loss: 0.1368


In [68]:
model_rnn.save("simple_rnn.keras")

In [75]:
model_lstm = Sequential()

model_lstm.add(
    Embedding(
        input_dim=MAX_NB_WORDS,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH
    )
)

model_lstm.add(LSTM(128))
model_lstm.add(Dropout(0.3))
model_lstm.add(Dense(64, activation="relu"))
model_lstm.add(Dense(6, activation="sigmoid"))

model_lstm.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [71]:
history_lstm = model_lstm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 364s 323ms/step - accuracy: 0.9663 - loss: 0.1454 - val_accuracy: 0.9937 - val_loss: 0.1380
Epoch 2/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 384s 342ms/step - accuracy: 0.9921 - loss: 0.0928 - val_accuracy: 0.9937 - val_loss: 0.0565
Epoch 3/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 372s 331ms/step - accuracy: 0.9940 - loss: 0.0480 - val_accuracy: 0.9937 - val_loss: 0.0489
Epoch 4/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 388s 346ms/step - accuracy: 0.9942 - loss: 0.0411 - val_accuracy: 0.9937 - val_loss: 0.0499
Epoch 5/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 373s 333ms/step - accuracy: 0.9923 - loss: 0.0366 - val_accuracy: 0.9937 - val_loss: 0.0527


In [76]:
model_lstm.save("lstm.keras")

In [77]:
model_bilstm = Sequential()

model_bilstm.add(
    Embedding(
        input_dim=MAX_NB_WORDS,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH
    )
)

model_bilstm.add(Bidirectional(LSTM(128)))
model_bilstm.add(Dropout(0.3))
model_bilstm.add(Dense(64, activation="relu"))
model_bilstm.add(Dropout(0.2))
model_bilstm.add(Dense(6, activation="sigmoid"))

model_bilstm.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [78]:
history_bilstm = model_bilstm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 595s 526ms/step - accuracy: 0.8660 - loss: 0.0770 - val_accuracy: 0.9937 - val_loss: 0.0511
Epoch 2/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 576s 514ms/step - accuracy: 0.9706 - loss: 0.0481 - val_accuracy: 0.9937 - val_loss: 0.0482
Epoch 3/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 581s 518ms/step - accuracy: 0.9803 - loss: 0.0418 - val_accuracy: 0.9937 - val_loss: 0.0487
Epoch 4/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 574s 512ms/step - accuracy: 0.9713 - loss: 0.0377 - val_accuracy: 0.8424 - val_loss: 0.0518


In [80]:
model_bilstm.save('bilstm.keras')

In [81]:
test_pred = model_bilstm.predict(X_test)

4787/4787 ━━━━━━━━━━━━━━━━━━━━ 162s 34ms/step


In [82]:
test_pred

array([[9.97109771e-01, 4.10020858e-01, 9.42539215e-01, 7.63404816e-02,
        8.54539096e-01, 2.42239773e-01],
       [3.55716795e-04, 1.88204385e-08, 2.72506331e-05, 1.99655278e-06,
        3.36274970e-05, 3.12271800e-06],
       [1.01191718e-02, 4.56815178e-06, 8.60348344e-04, 1.25383769e-04,
        1.32771151e-03, 1.67764214e-04],
       ...,
       [1.48128462e-03, 1.32616151e-07, 1.07432308e-04, 7.75829085e-06,
        1.36077317e-04, 1.47095516e-05],
       [1.47532916e-03, 1.44444414e-07, 1.07369822e-04, 8.42565987e-06,
        1.40910357e-04, 1.50605265e-05],
       [9.76726830e-01, 9.65780467e-02, 7.26093411e-01, 3.88435349e-02,
        6.77582562e-01, 1.20862722e-01]], shape=(153164, 6), dtype=float32)

In [95]:
lstm_pred = model_lstm.predict(X_test) 

4787/4787 ━━━━━━━━━━━━━━━━━━━━ 116s 24ms/step


In [97]:
lstm_labels = (lstm_pred>=0.5).astype(int)
lstm_labels[0]

array([1, 1, 1, 1, 1, 0])

In [83]:
pred_labels = (test_pred >= 0.5).astype(int)

In [89]:
pred_labels[0]

array([1, 0, 1, 0, 1, 0])

In [88]:
sentences_test[0]

'yo bitch ja rule is more succesful then you will ever be whats up with you and hating you sad mofuckas i should bitch slap ur pethedic white faces and get you to kiss my ass you guys sicken me ja rule is about pride in da music man dont diss that shit on him and nothin is wrong bein like tupac he was a brother too fuckin white boys get things right next time'

In [91]:
classes

['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

In [92]:
test_df['toxic'] = pred_labels[:,0]
test_df['severe_toxic'] = pred_labels[:,1]
test_df['obscene'] = pred_labels[:,2]
test_df['threat'] = pred_labels[:,3]
test_df['insult'] = pred_labels[:,4]
test_df['identity_hate'] = pred_labels[:,5]

In [93]:
test_df.to_csv('test_predictions.csv', index=False)

In [94]:
test_df

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,yo bitch ja rule is more succesful then you wi...,1,0,1,0,1,0
1,0000247867823ef7,= = from rfc = = the title is fine as it is imo,0,0,0,0,0,0
2,00013b17ad220c46,= = sources = = zawe ashton on lapland,0,0,0,0,0,0
3,00017563c3f7919a,if you have a look back at the source the info...,0,0,0,0,0,0
4,00017695ad8997eb,i do not anonymously edit articles at all,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
153159,fffcd0960ee309b5,i totally agree this stuff is nothing but too ...,1,0,0,0,0,0
153160,fffd7a9a6eb32c16,= = throw from out field to home plate = = doe...,0,0,0,0,0,0
153161,fffda9e8d6fafa9e,= = okinotorishima categories = = i see your c...,0,0,0,0,0,0
153162,fffe8f1340a79fc2,= = one of the founding nations of the eu - ge...,0,0,0,0,0,0
